# 🎯 Classification Models - Amaliy Mashg'ulot

Bu notebookda biz 4 ta Classification algoritmini amalda qo'llaymiz:
1. **Logistic Regression**
2. **k-Nearest Neighbors (k-NN)**
3. **Decision Tree**
4. **Random Forest**

---

In [ ]:
# Kutubxonalarni import qilish
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Vizualizatsiya sozlamalari
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Warning'larni o'chirish
import warnings
warnings.filterwarnings('ignore')

print("✅ Barcha kutubxonalar muvaffaqiyatli import qilindi!")

---

## 📊 Ma'lumotlar to'plamini yuklash

Biz **Breast Cancer Wisconsin** ma'lumotlar to'plamidan foydalanamiz. Bu dataset ko'krak saratonini aniqlash uchun ishlatiladi.

- **569** namuna
- **30** feature
- **2** class: Malignant (yomon), Benign (yaxshi)

In [ ]:
# Breast Cancer datasetini yuklash
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

print("Dataset o'lchami:", X.shape)
print("\nFeature'lar:")
print(X.columns.tolist())
print("\nTarget classes:")
print(dict(zip([0, 1], cancer.target_names)))
print("\nClass Distribution:")
print(y.value_counts())
print(f"\n{cancer.target_names[1]}: {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)")
print(f"{cancer.target_names[0]}: {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)")

In [ ]:
# Birinchi 5 qatorni ko'rish
X.head()

In [ ]:
# Ma'lumotlarni vizualizatsiya qilish (dastlabki 2 feature)
plt.figure(figsize=(10, 6))
plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, cmap='coolwarm', alpha=0.7, edgecolors='k')
plt.xlabel(cancer.feature_names[0], fontsize=12)
plt.ylabel(cancer.feature_names[1], fontsize=12)
plt.title('Breast Cancer Dataset - Feature Scatter', fontsize=14, fontweight='bold')
plt.colorbar(label='Target (0=Malignant, 1=Benign)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🔧 Ma'lumotlarni tayyorlash

Train va Test to'plamlariga ajratish va scaling qilish.

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set o'lchami:", X_train.shape)
print("Test set o'lchami:", X_test.shape)
print("\nTrain set class distribution:")
print(y_train.value_counts())
print("\nTest set class distribution:")
print(y_test.value_counts())

In [ ]:
# Feature Scaling (k-NN va Logistic Regression uchun kerak)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Scaling amalga oshirildi")
print("\nOriginal feature'lar (dastlabki 5):")
print(X_train.iloc[0, :5].values)
print("\nScaled feature'lar (dastlabki 5):")
print(X_train_scaled[0, :5])

---

# 1️⃣ Logistic Regression

### Vazifa:
Logistic Regression modelini yarating va baholang.

In [ ]:
# Logistic Regression modelini yaratish
log_reg = LogisticRegression(random_state=42, max_iter=10000)

# Modelni o'rgatish
log_reg.fit(X_train_scaled, y_train)

# Bashorat qilish
y_pred_log = log_reg.predict(X_test_scaled)
y_pred_proba_log = log_reg.predict_proba(X_test_scaled)[:, 1]

print("✅ Logistic Regression modeli o'rgatildi!")

In [ ]:
# Model baholash
print("="*60)
print("LOGISTIC REGRESSION - NATIJALAR")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_log):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_log):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_log):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_log):.4f}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log, target_names=cancer.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_log)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=cancer.target_names, 
            yticklabels=cancer.target_names, cbar=True)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - Logistic Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_log)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Logistic Regression', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# 2️⃣ k-Nearest Neighbors (k-NN)

### Vazifa:
Eng yaxshi k qiymatini toping va k-NN modelini baholang.

In [ ]:
# Eng yaxshi k qiymatini topish
k_values = range(1, 31)
test_scores = []
train_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Best k
best_k = k_values[np.argmax(test_scores)]
print(f"\n🎯 Eng yaxshi k qiymati: {best_k}")
print(f"Test accuracy: {max(test_scores):.4f}")

In [ ]:
# k qiymatiga qarab accuracy
plt.figure(figsize=(12, 6))
plt.plot(k_values, train_scores, label='Train Accuracy', marker='o', linewidth=2)
plt.plot(k_values, test_scores, label='Test Accuracy', marker='s', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', linewidth=2, label=f'Best k={best_k}')
plt.xlabel('k (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('k-NN: Accuracy vs k', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Eng yaxshi k bilan model yaratish
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print("="*60)
print(f"k-NN (k={best_k}) - NATIJALAR")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_knn):.4f}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn, target_names=cancer.target_names))

In [ ]:
# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', 
            xticklabels=cancer.target_names, 
            yticklabels=cancer.target_names, cbar=True)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Confusion Matrix - k-NN (k={best_k})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# 3️⃣ Decision Tree

### Vazifa:
Decision Tree modelini yarating, vizualizatsiya qiling va baholang.

In [ ]:
# Decision Tree modelini yaratish
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("="*60)
print("DECISION TREE - NATIJALAR")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=cancer.target_names))

In [ ]:
# Decision Tree Visualization
plt.figure(figsize=(25, 12))
plot_tree(dt, 
          feature_names=cancer.feature_names,
          class_names=cancer.target_names,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Visualization - Breast Cancer', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': cancer.feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15], color='skyblue')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Feature Importance - Decision Tree', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 10 Feature Importance:")
print(feature_importance.head(10))

In [ ]:
# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Purples', 
            xticklabels=cancer.target_names, 
            yticklabels=cancer.target_names, cbar=True)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - Decision Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔍 Decision Tree Hyperparameter Tuning

GridSearchCV yordamida eng yaxshi parametrlarni topamiz.

In [ ]:
# Hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🎯 Eng yaxshi parametrlar:")
print(grid_search.best_params_)
print(f"\n📊 Best Cross-Validation Score: {grid_search.best_score_:.4f}")
print(f"📊 Test Score: {grid_search.score(X_test, y_test):.4f}")

---

# 4️⃣ Random Forest

### Vazifa:
Random Forest modelini yarating va baholang.

In [ ]:
# Random Forest modelini yaratish
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("="*60)
print("RANDOM FOREST - NATIJALAR")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=cancer.target_names))

In [ ]:
# Feature Importance (Random Forest)
feature_importance_rf = pd.DataFrame({
    'feature': cancer.feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance_rf['feature'][:15], feature_importance_rf['importance'][:15], color='coral')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 10 Feature Importance:")
print(feature_importance_rf.head(10))

In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=cancer.target_names, 
            yticklabels=cancer.target_names, cbar=True)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔍 Random Forest Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print("\n🎯 Eng yaxshi parametrlar:")
print(grid_search_rf.best_params_)
print(f"\n📊 Best Cross-Validation Score: {grid_search_rf.best_score_:.4f}")
print(f"📊 Test Score: {grid_search_rf.score(X_test, y_test):.4f}")

---

# 📊 Barcha Modellarni Taqqoslash

In [ ]:
# Barcha modellar natijalari
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'k-NN', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision': [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
})

print("\n" + "="*80)
print("BARCHA MODELLAR TAQQOSLASH")
print("="*80)
print(results.to_string(index=False))
print("="*80)

# Eng yaxshi model
best_model = results.loc[results['Accuracy'].idxmax(), 'Model']
best_accuracy = results['Accuracy'].max()
print(f"\n🏆 Eng yaxshi model: {best_model} (Accuracy: {best_accuracy:.4f})")

In [ ]:
# Visualization: Model Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['skyblue', 'lightgreen', 'lightcoral', 'orange']

for i, (ax, metric, color) in enumerate(zip(axes.flat, metrics, colors)):
    ax.bar(results['Model'], results[metric], color=color, alpha=0.7, edgecolor='black')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0.85, 1.0)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for j, v in enumerate(results[metric]):
        ax.text(j, v + 0.005, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---

# 🎓 Qo'shimcha Vazifalar

## Vazifa 1: Wine Dataset bilan ishlash

Wine dataset (3 class) uchun barcha 4 ta modelni qo'llang va taqqoslang.

In [ ]:
# Wine datasetini yuklash
wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = pd.Series(wine.target)

print("Wine Dataset o'lchami:", X_wine.shape)
print("\nClass Distribution:")
print(y_wine.value_counts())
print("\nTarget names:", wine.target_names)

# TODO: Train-test split qiling
# TODO: Barcha 4 ta modelni o'rgating
# TODO: Natijalarni taqqoslang

## Vazifa 2: Cross-Validation

Barcha modellar uchun 5-fold cross-validation qiling va o'rtacha natijalarni ko'rsating.

In [ ]:
# TODO: Cross-validation implementation
# Hint: cross_val_score() dan foydalaning

from sklearn.model_selection import cross_val_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=10000),
    'k-NN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}

# TODO: Har bir model uchun cross-validation qiling

---

# 📝 Xulosa

Siz ushbu amaliy mashg'ulotda:

1. ✅ **Logistic Regression** - Binary classification
2. ✅ **k-NN** - Instance-based learning
3. ✅ **Decision Tree** - Tree-based model
4. ✅ **Random Forest** - Ensemble method

Har bir modelni:
- O'rgatdingiz
- Baholadingiz (Accuracy, Precision, Recall, F1-Score)
- Vizualizatsiya qildingiz
- Hyperparameter tuning qildingiz
- Taqqosladingiz

🎉 **Tabriklaymiz! Classification Models bo'yicha amaliy ko'nikmalaringiz rivojlandi!**